In [ ]:
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

pip('mediapipe', 'protobuf==5.29.4', 'opencv-python-headless',
    'xgboost', 'shap', 'scipy', 'pyyaml', 'openpyxl',
    'torch', 'torchvision', 'scikit-learn', 'ipywidgets')

import mediapipe as mp
print('mediapipe:', mp.__version__)
print('Install complete — restart runtime/kernel before running Cell 2')

mediapipe: 0.10.35
Install complete — restart runtime/kernel before running Cell 2


In [ ]:
# %% ════════════════════════════════════════════════════════════════
# CELL 2 — IMPORTS + ALL CONSTANTS
# Single source of truth — no constant is defined anywhere else.
# Run this cell after every restart before running anything else.
# ════════════════════════════════════════════════════════════════

import os, json, time, copy, pickle, urllib.request, logging
from datetime import datetime
from typing import List, Optional
from collections import Counter

import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from scipy.signal import savgol_filter, find_peaks
from scipy.interpolate import interp1d
from sklearn.metrics import accuracy_score, f1_score, classification_report
from scipy.stats import pearsonr

# ── Environment detection ────────────────────────────────────────────
IN_COLAB = False
try:
    import google.colab  # noqa
    IN_COLAB = True
    from google.colab import drive
    drive.mount('/content/drive')
    print('Running in Colab — Drive mounted')
except ImportError:
    print('Running in VS Code / local Jupyter')

# ── Device ───────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

# ── Data paths ───────────────────────────────────────────────────────
# Edit these to match where your CSV files actually are.
if IN_COLAB:
    DATA_DIR   = '/content/drive/MyDrive/datasets'
    MODEL_SAVE = '/content/drive/MyDrive/models/rehabnet_best.pth'
else:
    DATA_DIR   = './datasets'          # local folder next to this file
    MODEL_SAVE = './models/rehabnet_best.pth'

UIPRMD_CSV = os.path.join(DATA_DIR, 'uiprmd.csv')
KIMORE_CSV = os.path.join(DATA_DIR, 'kimore.csv')
KERAAL_CSV = os.path.join(DATA_DIR, 'keraal.csv')

os.makedirs(os.path.dirname(MODEL_SAVE), exist_ok=True)

# ── Sequence constants ────────────────────────────────────────────────
TARGET_LEN  = 150
N_JOINTS    = 6
IN_CHANNELS = 3
MIN_FRAMES  = 20
FPS         = 30
N_EPOCHS    = 60
EPS         = 1e-6

# ── Joint order ───────────────────────────────────────────────────────
JOINT_ORDER = ['l_hip', 'l_knee', 'l_ankle', 'r_hip', 'r_knee', 'r_ankle']

# ── Exercise list ─────────────────────────────────────────────────────
# SINGLE definition — RehabNet._N_EXERCISES must equal len(EXERCISE_LIST)
EXERCISE_LIST = [
    'deep_squat', 'hurdle_step', 'inline_lunge', 'side_lunge',
    'sit_to_stand', 'straight_leg_raise',
    'squat', 'ctk_squat', 'knee_bend', 'unknown'
]
EXERCISE2IDX = {e: i for i, e in enumerate(EXERCISE_LIST)}
N_EXERCISES  = len(EXERCISE_LIST)   # 10  ← matches RehabNet._N_EXERCISES below

# ── Hardware modes ────────────────────────────────────────────────────
HARDWARE_MODES = {
    0: {'name': 'assist', 'scale': -0.3, 'bias': 0.5, 'min': 0.0, 'max': 0.8},
    1: {'name': 'resist', 'scale':  0.6, 'bias': 0.1, 'min': 0.0, 'max': 0.8},
}
N_MODES = len(HARDWARE_MODES)   # 2

# ── Rep segmentation: minimum inter-peak distance per exercise ─────────
EX_MIN_DIST = {
    'inline_lunge': 60, 'side_lunge': 60, 'deep_squat': 45,
    'sit_to_stand': 80, 'straight_leg_raise': 40,
    'knee_bend': 50, 'squat': 45, 'ctk_squat': 45,
    'hurdle_step': 45, 'unknown': 50, 'default': 50,
}

# ── Deficit targets per exercise (degrees) ────────────────────────────
TARGET_ANGLES = {
    'deep_squat': 90., 'inline_lunge': 90., 'side_lunge': 90.,
    'sit_to_stand': 100., 'straight_leg_raise': 170.,
    'knee_bend': 90., 'squat': 90., 'default': 90.,
}

# ── Feedback rules (priority-ordered, highest = most important) ────────
# scalars index map:
#   0=l_rom/180  1=r_rom/180  2=l_peak/180  3=r_peak/180
#   4=sym/100    5=vel/200    6=jerk/10     7=trunk
#   8=lag/150    9=smooth
FEEDBACK_RULES = [
    {'name': 'insufficient_ROM',   'priority': 10,
     'check':   lambda s: min(s[0], s[1]) * 180 < 50,
     'message': 'Bend your knee a little further — try to reach 90°.',
     'short':   'Bend knee further'},
    {'name': 'too_fast',           'priority': 9,
     'check':   lambda s: s[5] * 200 > 80,
     'message': 'Slow down — take about 3 seconds for each bend.',
     'short':   'Move slower'},
    {'name': 'high_jerk',          'priority': 8,
     'check':   lambda s: s[6] * 10 > 8,
     'message': "Don't jerk your knee — keep the movement smooth.",
     'short':   'Avoid jerking'},
    {'name': 'trunk_compensation', 'priority': 7,
     'check':   lambda s: s[7] > 0.35,
     'message': 'Keep your back straight — try not to lean forward.',
     'short':   'Keep back straight'},
    {'name': 'asymmetric',         'priority': 6,
     'check':   lambda s: s[4] * 100 > 25,
     'message': 'Try to put equal weight on both legs.',
     'short':   'Equal weight both legs'},
    {'name': 'temporal_lag',       'priority': 5,
     'check':   lambda s: s[8] * 150 > 15,
     'message': 'Keep both legs moving at the same time.',
     'short':   'Sync both legs'},
]

# ── MediaPipe model path ───────────────────────────────────────────────
MP_MODEL_PATH = '/tmp/pose_landmarker.task'
MP_MODEL_URL  = ('https://storage.googleapis.com/mediapipe-models/'
                 'pose_landmarker/pose_landmarker_heavy/float16/'
                 'latest/pose_landmarker_heavy.task')

# ── iTransformer / Dataset-embedding constants ─────────────────────
D_MODEL           = 128
N_HEADS           = 4
N_LAYERS          = 2
D_FF              = 256
ITRANS_DROPOUT    = 0.1
DATASET_SOURCE_MAP = {'uiprmd': 0, 'kimore': 1, 'keraal': 2, 'patient_video': 3}
N_DATASETS         = len(DATASET_SOURCE_MAP)   # 4
DATASET_EMBED_DIM  = 16
N_SCALARS          = 10

print(f'Constants loaded. N_EXERCISES={N_EXERCISES}  N_MODES={N_MODES}  N_DATASETS={N_DATASETS}')

Mounted at /content/drive
Running in Colab — Drive mounted
Device: cuda
Constants loaded. N_EXERCISES=10  N_MODES=2  N_DATASETS=4


In [ ]:
# %% ════════════════════════════════════════════════════════════════
# CELL 3 — ALL FUNCTIONS
# Paste this entire block as one cell.  Order matters inside here;
# do not split it across cells.
# Sections:
#   A. Shared helpers
#   B. PS1  (video → DataFrame)
#   C. Keypoint processing  (DataFrame → tensors)
#   D. Bridge  (DataFrame → per-rep sample dicts)
#   E. Dataset loaders  (load_uiprmd / load_kimore / load_keraal)
#   F. Model  (STGCNBlock → STGCN → ClinicalTransformer → RehabNet)
#   G. PyTorch Dataset + DataLoader helpers
#   H. Training  (compute_class_weights, _train_epoch, _evaluate, run_loso)
#   I. Inference + feedback  (compute_torque, infer_sample, generate_feedback)
#   J. Patient session  (run_patient_video)
# ════════════════════════════════════════════════════════════════

# ─────────────────────────────────────────────────────────────────
# A. SHARED HELPERS
# ─────────────────────────────────────────────────────────────────

def safe_np(x, fill=0.0, clip=None, dtype=np.float32):
    x = np.asarray(x, dtype=dtype)
    x = np.nan_to_num(x, nan=fill, posinf=fill, neginf=fill)
    if clip is not None:
        x = np.clip(x, -clip, clip)
    return x.astype(dtype)

def valid_np(x):
    x = np.asarray(x)
    return x.size > 0 and np.isfinite(x).all()

def _smooth(arr, win=7, poly=2):
    arr = safe_np(arr)
    if len(arr) < win + 2:
        return arr
    w = win if win % 2 == 1 else win + 1
    w = max(w, poly + 2 if (poly + 2) % 2 == 1 else poly + 3)
    try:
        return safe_np(savgol_filter(arr, w, poly))
    except Exception:
        return arr

def _resample(arr, n):
    arr = safe_np(arr)
    if len(arr) == n:
        return arr
    if len(arr) < 2:
        return np.zeros(n, dtype=np.float32)
    return safe_np(np.interp(np.linspace(0, 1, n),
                              np.linspace(0, 1, len(arr)), arr))

def _fill_missing(kps):
    kps = safe_np(kps)
    T, J, C = kps.shape
    for j in range(J):
        for c in range(C):
            col = kps[:, j, c]
            z   = (~np.isfinite(col)) | (col == 0.0)
            if z.all():
                opp = j + 3 if j < 3 else j - 3
                kps[:, j, c] = kps[:, opp, c]
            elif z.any() and (~z).sum() >= 2:
                f = interp1d(np.where(~z)[0], col[~z],
                             bounds_error=False, fill_value='extrapolate')
                kps[:, j, c] = f(np.arange(T))
    return safe_np(kps)

def _normalise_kps(kps):
    kps  = safe_np(kps)
    hip  = ((kps[:, 0, :] + kps[:, 3, :]) / 2.0).mean(axis=0)
    kps  = kps - hip
    l    = np.mean(np.linalg.norm(kps[:, 0, :] - kps[:, 2, :], axis=1))
    r    = np.mean(np.linalg.norm(kps[:, 3, :] - kps[:, 5, :], axis=1))
    scale = (l + r) / 2.0
    if not np.isfinite(scale) or scale < 1e-4:
        scale = 1.0
    return safe_np(np.clip(kps / scale, -5.0, 5.0), clip=5.0)

def build_adjacency():
    A = np.zeros((N_JOINTS, N_JOINTS), dtype=np.float32)
    for i, j in [(0, 1), (1, 2), (3, 4), (4, 5), (0, 3), (1, 4), (2, 5)]:
        A[i, j] = A[j, i] = 1.0
    np.fill_diagonal(A, 1.0)
    d = np.diag(1.0 / np.sqrt(A.sum(axis=1) + EPS))
    return (d @ A @ d).astype(np.float32)

ADJ = build_adjacency()

# ─────────────────────────────────────────────────────────────────
# B. PS1 — VIDEO → DATAFRAME
# ─────────────────────────────────────────────────────────────────

MIN_DETECT_CONF = 0.5
MIN_TRACK_CONF  = 0.5

JOINT_COLS = []
for _jn in ['HipLeft', 'KneeLeft', 'AnkleLeft', 'HipRight', 'KneeRight', 'AnkleRight']:
    for _ax in ['x', 'y', 'z']:
        JOINT_COLS.append(f'{_jn}_{_ax}')
BASE_COLS = ['subject_id', 'session', 'label', 'frame_id', 'time_s', 'detection']

def ps1_prepare_frame(frame, target_w=640):
    h, w = frame.shape[:2]
    if w > target_w:
        frame = cv2.resize(frame, (target_w, int(h * target_w / w)))
    return frame

def ps1_extract_row(landmarks):
    mp_map = {
        'HipLeft': 23, 'KneeLeft': 25, 'AnkleLeft': 27,
        'HipRight': 24, 'KneeRight': 26, 'AnkleRight': 28,
    }
    row = {}
    for jname, idx in mp_map.items():
        lm = landmarks[idx]
        row[f'{jname}_x'] = float(lm.x)
        row[f'{jname}_y'] = float(lm.y)
        row[f'{jname}_z'] = float(lm.z)
    return row

def ps1_root_center(df):
    for ax in ['x', 'y', 'z']:
        mid = (df[f'HipLeft_{ax}'] + df[f'HipRight_{ax}']) / 2.0
        for jn in ['HipLeft', 'KneeLeft', 'AnkleLeft',
                   'HipRight', 'KneeRight', 'AnkleRight']:
            df[f'{jn}_{ax}'] = df[f'{jn}_{ax}'] - mid
    return df

def ps1_bone_normalize(df):
    for side, (h, k, a) in [
        ('L', ('HipLeft',  'KneeLeft',  'AnkleLeft')),
        ('R', ('HipRight', 'KneeRight', 'AnkleRight')),
    ]:
        for ax in ['x', 'y', 'z']:
            thigh = np.sqrt(((df[f'{h}_{ax}'] - df[f'{k}_{ax}']) ** 2).mean()) + EPS
            for jname in [h, k, a]:
                df[f'{jname}_{ax}'] = df[f'{jname}_{ax}'] / thigh
    return df

def _knee_angle_from_df(df, side='L'):
    prefix = {
        'L': ('HipLeft',  'KneeLeft',  'AnkleLeft'),
        'R': ('HipRight', 'KneeRight', 'AnkleRight'),
    }[side]
    h  = df[[f'{prefix[0]}_{ax}' for ax in 'xyz']].values
    k  = df[[f'{prefix[1]}_{ax}' for ax in 'xyz']].values
    a  = df[[f'{prefix[2]}_{ax}' for ax in 'xyz']].values
    v1 = h - k
    v2 = a - k
    cos = (np.sum(v1 * v2, axis=1) /
           (np.linalg.norm(v1, axis=1) * np.linalg.norm(v2, axis=1) + EPS))
    return np.degrees(np.arccos(np.clip(cos, -1, 1)))

def ps1_extract_features(df):
    df['angle_knee_L']        = _knee_angle_from_df(df, 'L')
    df['angle_knee_R']        = _knee_angle_from_df(df, 'R')
    df['angle_knee_L_smooth'] = _smooth(df['angle_knee_L'].values)
    df['angle_knee_R_smooth'] = _smooth(df['angle_knee_R'].values)
    return df

def ps1_clean_nulls(df):
    return df.ffill().bfill().fillna(0.0)

def _ensure_mp_model():
    if not os.path.exists(MP_MODEL_PATH):
        print('  Downloading MediaPipe pose model (~30 MB)...')
        urllib.request.urlretrieve(MP_MODEL_URL, MP_MODEL_PATH)
        print('  Downloaded.')

def ps1_process_video(video_path, patient_id,
                      exercise='unknown', session='1', label='unknown'):
    """
    VIDEO FILE → processed DataFrame  (one row per detected frame).
    Works on any mp4/avi/webm file.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise FileNotFoundError(f'Cannot open video: {video_path}')

    fps_vid    = cap.get(cv2.CAP_PROP_FPS) or FPS
    rows, frame_idx, detected = [], 0, 0
    t0 = time.time()

    _ensure_mp_model()

    import mediapipe as mp
    from mediapipe.tasks import python as _mptasks
    from mediapipe.tasks.python import vision as _mpvision

    opts = _mpvision.PoseLandmarkerOptions(
        base_options=_mptasks.BaseOptions(model_asset_path=MP_MODEL_PATH),
        running_mode=_mpvision.RunningMode.VIDEO,
        num_poses=1,
        min_pose_detection_confidence=MIN_DETECT_CONF,
        min_pose_presence_confidence=MIN_DETECT_CONF,
        min_tracking_confidence=MIN_TRACK_CONF,
        output_segmentation_masks=False,
    )

    frame_step_ms = int(1000.0 / fps_vid)

    with _mpvision.PoseLandmarker.create_from_options(opts) as pose:
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            frame_idx += 1
            frame_ms   = frame_idx * frame_step_ms
            prepped    = ps1_prepare_frame(frame)
            rgb        = cv2.cvtColor(prepped, cv2.COLOR_BGR2RGB)
            mp_img     = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            res        = pose.detect_for_video(mp_img, frame_ms)

            row = {'subject_id': patient_id, 'session': session,
                   'label': label, 'frame_id': frame_idx,
                   'time_s': round(frame_idx / fps_vid, 4), 'detection': 0}
            row.update({col: float('nan') for col in JOINT_COLS})

            if res.pose_landmarks and len(res.pose_landmarks) > 0:
                detected += 1
                row['detection'] = 1
                row.update(ps1_extract_row(res.pose_landmarks[0]))
            rows.append(row)

    cap.release()

    if frame_idx == 0:
        raise ValueError('Video is empty')

    det_pct = 100 * detected / frame_idx
    print(f'  PS1: {frame_idx} frames  detection={det_pct:.0f}%  '
          f'{time.time()-t0:.1f}s')
    if det_pct < 20:
        print(f'  WARNING: low detection ({det_pct:.0f}%) — '
              f'check lighting and camera angle')

    df = pd.DataFrame(rows, columns=BASE_COLS + JOINT_COLS)
    df = df[df['detection'] == 1].reset_index(drop=True)
    df = ps1_root_center(df)
    df = ps1_bone_normalize(df)
    df = ps1_extract_features(df)
    df = ps1_clean_nulls(df)
    return df


# ─────────────────────────────────────────────────────────────────
# C. KEYPOINT PROCESSING  (shared by PS1 pipeline and dataset loaders)
# ─────────────────────────────────────────────────────────────────

def process_keypoints(raw):
    """
    raw: np.ndarray shape (T, N_JOINTS, 3)
    Returns: np.ndarray shape (TARGET_LEN, N_JOINTS, 3), normalised
    """
    raw = safe_np(raw)
    if raw.ndim != 3 or raw.shape[1] != N_JOINTS or raw.shape[2] != 3:
        raise ValueError(f'Expected (T,{N_JOINTS},3), got {raw.shape}')
    if len(raw) < MIN_FRAMES:
        raise ValueError(f'Too short: {len(raw)} frames (min={MIN_FRAMES})')
    for j in range(N_JOINTS):
        for c in range(3):
            raw[:, j, c] = _smooth(raw[:, j, c])
    raw = _fill_missing(raw)
    out = np.zeros((TARGET_LEN, N_JOINTS, 3), dtype=np.float32)
    for j in range(N_JOINTS):
        for c in range(3):
            out[:, j, c] = _resample(raw[:, j, c], TARGET_LEN)
    return safe_np(_normalise_kps(out), clip=5.0)

def extract_scalars(kps):
    """
    kps: np.ndarray (TARGET_LEN, N_JOINTS, 3)
    Returns: np.ndarray (10,) normalised scalar features
    """
    kps = safe_np(kps, clip=5.0)

    def _ang(a, b, c):
        ba, bc = a - b, c - b
        d = np.maximum(np.linalg.norm(ba, 1) * np.linalg.norm(bc, 1), EPS)
        return np.degrees(np.arccos(np.clip(np.sum(ba * bc, 1) / d, -1, 1)))

    lk  = _ang(kps[:, 0, :], kps[:, 1, :], kps[:, 2, :])
    rk  = _ang(kps[:, 3, :], kps[:, 4, :], kps[:, 5, :])
    lr  = float(np.nanmax(lk) - np.nanmin(lk))
    rr  = float(np.nanmax(rk) - np.nanmin(rk))
    sym = abs(lr - rr) / ((lr + rr) / 2 + EPS) * 100 if (lr + rr) > 1 else 0.0
    vel = float(np.max(np.abs(np.diff(lk))) * FPS) if len(lk) > 1 else 0.0

    hip  = (kps[:, 0, :] + kps[:, 3, :]) / 2.0
    jk   = np.diff(np.diff(hip, axis=0), axis=0)
    jerk = float(np.nanmean(np.linalg.norm(jk, axis=1))) if len(jk) else 0.0

    pk    = int(np.argmin(lk))
    trunk = float(abs(hip[pk, 0] - hip[0, 0]))

    lkc, rkc = lk - np.nanmean(lk), rk - np.nanmean(rk)
    lag = float(abs(
        np.argmax(np.correlate(lkc, rkc, 'full')) - (len(lk) - 1)
    )) if np.nanstd(lkc) > EPS and np.nanstd(rkc) > EPS else 0.0

    smooth = float(np.clip(1.0 / (np.nanstd(np.diff(lk)) + EPS) / 100, 0, 1))

    return safe_np(np.array([
        np.clip(lr / 180,          0, 1),
        np.clip(rr / 180,          0, 1),
        np.clip(np.min(lk) / 180,  0, 1),
        np.clip(np.min(rk) / 180,  0, 1),
        np.clip(sym / 100,         0, 5),
        np.clip(vel / 200,         0, 5),
        np.clip(jerk / 10,         0, 5),
        np.clip(trunk,             0, 5),
        np.clip(lag / TARGET_LEN,  0, 1),
        smooth,
    ], dtype=np.float32), fill=0.0, clip=10.0)


# ─────────────────────────────────────────────────────────────────
# D. BRIDGE — DATAFRAME → PER-REP SAMPLE DICTS
# ─────────────────────────────────────────────────────────────────

def _df_to_kps(df):
    cols = []
    for jn in ['HipLeft', 'KneeLeft', 'AnkleLeft',
               'HipRight', 'KneeRight', 'AnkleRight']:
        cols += [f'{jn}_x', f'{jn}_y', f'{jn}_z']
    return df[cols].values.astype(np.float32).reshape(len(df), N_JOINTS, 3)

def _segment_reps(df, angle_col='angle_knee_L_smooth',
                  min_frames=MIN_FRAMES, min_rom=20.0, exercise='unknown'):
    """Split a continuous DataFrame into per-rep DataFrames."""
    if angle_col in df.columns:
        angles = df[angle_col].ffill().bfill().values
    else:
        angles = _knee_angle_from_df(df, 'L')

    n        = len(angles)
    min_dist = max(EX_MIN_DIST.get(exercise, EX_MIN_DIST['default']), n // 20)

    peaks, _ = find_peaks(
        -angles,
        distance=min_dist,
        prominence=min_rom * 0.6,
        width=min_dist // 3,
    )

    print(f'  Segmenter: n_frames={n}  min_dist={min_dist}  peaks={len(peaks)}')

    if len(peaks) == 0:
        return [df]

    mids   = [(peaks[i] + peaks[i + 1]) // 2 for i in range(len(peaks) - 1)]
    bounds = [0] + mids + [n]
    reps   = [df.iloc[bounds[i]:bounds[i + 1]].copy()
              for i in range(len(bounds) - 1)
              if len(df.iloc[bounds[i]:bounds[i + 1]]) >= min_frames]
    return reps if reps else [df]

def ps1_df_to_samples(df, patient_id, exercise='unknown',
                       label=-1, source='patient_video'):
    """PS1 DataFrame → list of sample dicts ready for RehabNet."""
    reps   = _segment_reps(df, exercise=exercise)
    ex_idx = EXERCISE2IDX.get(exercise, EXERCISE2IDX['unknown'])
    print(f'  Bridge: {len(reps)} rep segments → ', end='')
    samples = []
    for i, rep_df in enumerate(reps):
        try:
            kps = _df_to_kps(rep_df)
            kps = process_keypoints(kps)
            sc  = extract_scalars(kps)
            samples.append({
                'keypoints':    kps,
                'adj':          ADJ,
                'scalars':      sc,
                'label':        max(label, 0),
                'quality':      float(max(label, 0)),
                'exercise':     exercise,
                'exercise_idx': ex_idx,
                'subject':      patient_id,
                'source':       source,
                'rep_id':       f'{patient_id}_r{i + 1}',
            })
        except Exception as e:
            print(f'\n    Rep {i + 1} skipped: {e}')
    print(f'{len(samples)} valid')
    return samples


# ─────────────────────────────────────────────────────────────────
# E. DATASET LOADERS
# ─────────────────────────────────────────────────────────────────

def _kps_from_df_rows(df, xyz_cols):
    return (df[xyz_cols].values
            .astype(np.float32)
            .reshape(len(df), N_JOINTS, 3))

def load_uiprmd():
    if not os.path.exists(UIPRMD_CSV):
        print(f'UI-PRMD not found: {UIPRMD_CSV}')
        return []
    df       = pd.read_csv(UIPRMD_CSV)
    xyz_cols = [f'{j}_{ax}' for j in JOINT_ORDER for ax in ['x', 'y', 'z']]
    samples  = []
    for (subj, mov, ex_id, lv), grp in df.groupby(
            ['subject', 'movement', 'exercise_id', 'label']):
        grp = grp.sort_values('frame')
        raw = _kps_from_df_rows(grp, xyz_cols) / 1000.0
        try:
            kps = process_keypoints(raw)
            ex  = grp['exercise'].iloc[0]
            samples.append({
                'keypoints':    kps,
                'adj':          ADJ,
                'label':        int(lv),
                'quality':      float(lv),
                'exercise':     ex,
                'exercise_idx': EXERCISE2IDX.get(ex, EXERCISE2IDX['unknown']),
                'subject':      f'uiprmd_{subj}',
                'source':       'uiprmd',
                'scalars':      extract_scalars(kps),
            })
        except Exception as e:
            print(f'  skip uiprmd {subj}/{mov}/{ex_id}: {e}')
    print(f'UI-PRMD: {len(samples)} samples')
    return samples

def load_kimore():
    if not os.path.exists(KIMORE_CSV):
        print(f'KIMORE not found: {KIMORE_CSV}')
        return []
    df = pd.read_csv(KIMORE_CSV)
    xyz_cols = []
    for jn in ['HipLeft', 'KneeLeft', 'AnkleLeft',
               'HipRight', 'KneeRight', 'AnkleRight']:
        xyz_cols += [f'{jn}_x', f'{jn}_y', f'{jn}_z']
    df['quality_norm'] = df[['cTS', 'cPO', 'cCF']].mean(axis=1) / 50.0
    samples = []
    for subj, grp in df.groupby('Subject'):
        grp = grp.sort_values('FrameID')
        raw = _kps_from_df_rows(grp, xyz_cols)
        try:
            kps = process_keypoints(raw)
            samples.append({
                'keypoints':    kps,
                'adj':          ADJ,
                'label':        int(grp['label'].iloc[0]),
                'quality':      float(grp['quality_norm'].mean()),
                'exercise':     'squat',
                'exercise_idx': EXERCISE2IDX['squat'],
                'subject':      f'kimore_{subj}',
                'source':       'kimore',
                'scalars':      extract_scalars(kps),
            })
        except Exception as e:
            print(f'  skip kimore {subj}: {e}')
    print(f'KIMORE: {len(samples)} samples')
    return samples

def load_keraal():
    if not os.path.exists(KERAAL_CSV):
        print(f'Keraal not found: {KERAAL_CSV}')
        return []
    df = pd.read_csv(KERAAL_CSV)
    xyz_cols = []
    for jn in ['HipLeft', 'KneeLeft', 'AnkleLeft',
               'HipRight', 'KneeRight', 'AnkleRight']:
        xyz_cols += [f'{jn}_x', f'{jn}_y', f'{jn}_z']
    samples = []
    for ann_key, grp in df.groupby('ann_key'):
        grp = grp.sort_values('FrameID') if 'FrameID' in grp.columns else grp
        raw = _kps_from_df_rows(grp, xyz_cols)
        try:
            kps  = process_keypoints(raw)
            subj = '-'.join(str(ann_key).split('-')[:3])
            samples.append({
                'keypoints':    kps,
                'adj':          ADJ,
                'label':        int(grp['label'].iloc[0]),
                'quality':      float(grp['label'].iloc[0]),
                'exercise':     'ctk_squat',
                'exercise_idx': EXERCISE2IDX['ctk_squat'],
                'subject':      f'keraal_{subj}',
                'source':       'keraal',
                'scalars':      extract_scalars(kps),
            })
        except Exception as e:
            print(f'  skip keraal {ann_key}: {e}')
    print(f'Keraal: {len(samples)} samples')
    return samples

def build_master_dataset():
    raw   = load_uiprmd() + load_kimore() + load_keraal()
    clean = []
    for s in raw:
        try:
            kps = safe_np(s['keypoints'], clip=5.0)
            sc  = safe_np(s.get('scalars', extract_scalars(kps)), clip=10.0)
            q   = float(np.nan_to_num(s.get('quality', s.get('label', 1)),
                                      nan=float(s.get('label', 1))))
            ex  = int(s.get('exercise_idx', EXERCISE2IDX['unknown']))
            if kps.shape != (TARGET_LEN, N_JOINTS, 3):
                raise ValueError('bad kps shape')
            if sc.shape != (10,):
                raise ValueError('bad scalar shape')
            if not valid_np(kps) or not valid_np(sc):
                raise ValueError('NaN/Inf detected')
            if not (0 <= ex < N_EXERCISES):
                ex = EXERCISE2IDX['unknown']
            s.update({'keypoints': kps, 'scalars': sc,
                      'label':        int(s.get('label', 1)),
                      'quality':      float(np.clip(q, 0, 1)),
                      'exercise_idx': ex})
            clean.append(s)
        except Exception as e:
            print(f"  DROP {s.get('subject', '?')}: {e}")
    print(f'\nMaster dataset: {len(clean)} samples')
    print(f"  correct={sum(s['label']==1 for s in clean)}  "
          f"incorrect={sum(s['label']==0 for s in clean)}")
    print(f"  sources:   {set(s['source']   for s in clean)}")
    print(f"  exercises: {set(s['exercise'] for s in clean)}")
    return clean

def loso_split(samples, held_out):
    return ([s for s in samples if s['subject'] != held_out],
            [s for s in samples if s['subject'] == held_out])

def get_uiprmd_subjects(samples):
    return sorted({s['subject'] for s in samples if s['source'] == 'uiprmd'})


# ─────────────────────────────────────────────────────────────────
# F. MODEL  --  iTransformer transplant
#
#  OLD: STGCN -> ClinicalTransformer(time-step tokens) + BiLSTM -> FusionLayer
#  NEW: STGCN -> iTransformerEncoder(joint tokens + temporal attn pool)
#              -> FusionLayer(+ dataset-source embedding) -> same 3 heads
#
#  Caller interface UNCHANGED (all sections G-J work as before):
#    RehabNet.forward(kps, adj, scalars, mode=None, exercise_idx=None,
#                     dataset_id=None)
#      returns (logits[B,2], quality_raw[B], exercise_logits[B,N])
#    RehabNet.predict_exercise(kps, adj) -> (str, int)
# ─────────────────────────────────────────────────────────────────

# -- F0.  ST-GCN  (einsum-based graph conv, cleaner than original bmm) --

class STGCNBlock(nn.Module):
    def __init__(self, c_in, c_out, ks=9, stride=1):
        super().__init__()
        self.gcn_conv = nn.Conv2d(c_in, c_out, kernel_size=1)
        self.gcn_bn   = nn.BatchNorm2d(c_out)
        pad = (ks - 1) // 2
        self.tcn = nn.Sequential(
            nn.Conv2d(c_out, c_out, (ks, 1), (stride, 1), (pad, 0)),
            nn.BatchNorm2d(c_out),
        )
        self.relu    = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout(0.1)
        self.res = (nn.Sequential(nn.Conv2d(c_in, c_out, 1, (stride, 1)),
                                   nn.BatchNorm2d(c_out))
                    if c_in != c_out or stride != 1 else nn.Identity())

    def forward(self, x, A):
        if A.dim() == 3:
            A = A[0]
        res = self.res(x)
        xs  = torch.einsum('bctv,vw->bctw', x, A)
        xs  = self.relu(self.gcn_bn(self.gcn_conv(xs)))
        out = self.relu(self.tcn(xs) + res)
        return self.dropout(out)


class STGCN(nn.Module):
    def __init__(self):
        super().__init__()
        self.input_norm = nn.LayerNorm(IN_CHANNELS * N_JOINTS)
        self.b1 = STGCNBlock(3,  32)
        self.b2 = STGCNBlock(32, 64)
        self.b3 = STGCNBlock(64, 128)

    def forward(self, x, A):
        B, C, T, V = x.shape
        x  = torch.clamp(x, -5.0, 5.0)
        xf = x.permute(0, 2, 1, 3).contiguous().view(B * T, C * V)
        xf = self.input_norm(xf).view(B, T, C, V).permute(0, 2, 1, 3)
        return self.b3(self.b2(self.b1(xf, A), A), A)   # [B, 128, T, V]


# -- F1.  iTransformer building blocks (from itrans_v3) --

class TemporalAttentionPooling(nn.Module):
    # Learnable soft-attention over the T axis -> replaces mean(dim=2).
    # Input  [B, C, T, V]
    # Output [B, C, V]
    def forward(self, x):
        w = F.softmax(x.mean(dim=1, keepdim=True), dim=2)  # [B,1,T,V]
        return (x * w).sum(dim=2)                           # [B,C,V]


class iTransformerBlock(nn.Module):
    # Each JOINT is a token; attention runs across joints, not time.
    # Input / output: [B, V, d_model]
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.attn  = nn.MultiheadAttention(d_model, n_heads,
                                            dropout=dropout, batch_first=True)
        self.ff    = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.drop  = nn.Dropout(dropout)

    def forward(self, x):
        q = k = v = self.norm1(x)
        attn_out, attn_w = self.attn(q, k, v)
        x = x + self.drop(attn_out)
        x = x + self.drop(self.ff(self.norm2(x)))
        return x, attn_w


class iTransformerEncoder(nn.Module):
    # Input  [B, C, T, V]   (ST-GCN feature map)
    # Output [B, d_model]   (joint-interaction global representation)
    #
    # Pipeline:
    #   TemporalAttentionPooling  -> [B,C,V]
    #   permute + LayerNorm + clamp
    #   Linear C->d_model
    #   N x iTransformerBlock  (joint tokens attend to each other)
    #   mean-pool over V       -> [B, d_model]
    def __init__(self, in_ch,
                 d_model=D_MODEL, n_heads=N_HEADS,
                 n_layers=N_LAYERS, d_ff=D_FF, dropout=ITRANS_DROPOUT):
        super().__init__()
        self.temp_pool  = TemporalAttentionPooling()
        self.input_norm = nn.LayerNorm(in_ch)
        self.proj       = nn.Linear(in_ch, d_model)
        self.blocks     = nn.ModuleList([
            iTransformerBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])
        self._last_attn = []

    def forward(self, x):
        x = self.temp_pool(x)            # [B,C,V]
        x = x.permute(0, 2, 1)          # [B,V,C]
        x = self.input_norm(x)
        x = torch.clamp(x, -10.0, 10.0)
        x = self.proj(x)                 # [B,V,d_model]
        self._last_attn = []
        for blk in self.blocks:
            x, w = blk(x)
            self._last_attn.append(w.detach().cpu())
        return x.mean(dim=1)             # [B,d_model]


# -- F2.  FusionLayer with dataset-source embedding --

class FusionLayer(nn.Module):
    # Fuses: deep_rep + scalars + mode_emb + ex_emb + dataset_emb -> 128-d.
    # dataset_emb is new: learned correction for cross-dataset domain shift.
    def __init__(self, deep_dim=D_MODEL, scalar_dim=N_SCALARS,
                 n_modes=2, n_exercises=N_EXERCISES,
                 n_datasets=N_DATASETS, dataset_embed_dim=DATASET_EMBED_DIM,
                 out_dim=128, dropout=0.2):
        super().__init__()
        self.mode_emb    = nn.Embedding(n_modes,    16)
        self.ex_emb      = nn.Embedding(n_exercises, 8)
        self.dataset_emb = nn.Embedding(n_datasets, dataset_embed_dim)
        in_dim = deep_dim + scalar_dim + 16 + 8 + dataset_embed_dim
        self.net = nn.Sequential(
            nn.Linear(in_dim, out_dim), nn.LayerNorm(out_dim), nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(out_dim, out_dim), nn.LayerNorm(out_dim), nn.GELU(),
        )

    def forward(self, deep, scalars, mode, exercise_idx, dataset_id):
        cat = torch.cat([
            deep, scalars,
            self.mode_emb(mode),
            self.ex_emb(exercise_idx),
            self.dataset_emb(dataset_id),
        ], dim=-1)
        return self.net(cat)


# -- F3.  RehabNet  --  the transplanted heart --

class RehabNet(nn.Module):
    # Full pipeline: STGCN -> iTransformerEncoder -> FusionLayer -> 3 heads
    #
    # Transplant vs old base:
    #   REMOVED : ClinicalTransformer (time-step Transformer)
    #   REMOVED : BiLSTM branch
    #   ADDED   : iTransformerEncoder (joint-token attn + temporal attn pool)
    #   ADDED   : dataset_emb in FusionLayer (domain shift correction)
    #   UNCHANGED: 3-tuple return, predict_exercise, all callers in G-J
    _N_MODES     = 2
    _N_EXERCISES = 10   # must equal N_EXERCISES

    def __init__(self):
        super().__init__()
        self.stgcn  = STGCN()
        self.itrans = iTransformerEncoder(in_ch=128)
        self.fusion = FusionLayer()

        self.classify_head = nn.Sequential(
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.2), nn.Linear(64, 2))
        self.quality_head  = nn.Sequential(
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.2), nn.Linear(64, 1))
        self.ex_head       = nn.Sequential(
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, self._N_EXERCISES))

        self._init_weights()
        self._last_attn = None

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Linear, nn.Conv2d)):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, kps, adj, scalars, mode=None, exercise_idx=None,
                dataset_id=None):
        B, device = kps.shape[0], kps.device
        if mode         is None: mode         = torch.zeros(B, dtype=torch.long, device=device)
        if exercise_idx is None: exercise_idx = torch.zeros(B, dtype=torch.long, device=device)
        if dataset_id   is None: dataset_id   = torch.zeros(B, dtype=torch.long, device=device)
        mode         = torch.clamp(mode,         0, self._N_MODES     - 1)
        exercise_idx = torch.clamp(exercise_idx, 0, self._N_EXERCISES - 1)
        dataset_id   = torch.clamp(dataset_id,   0, N_DATASETS        - 1)

        kps     = torch.nan_to_num(kps,     nan=0., posinf=1., neginf=-1.)
        scalars = torch.nan_to_num(scalars, nan=0., posinf=1., neginf=-1.)

        feat_map        = self.stgcn(kps, adj)           # [B,128,T,V]
        global_rep      = self.itrans(feat_map)           # [B,128]
        self._last_attn = self.itrans._last_attn
        fused = self.fusion(global_rep, scalars, mode, exercise_idx, dataset_id)

        # Same 3-tuple as old RehabNet so every caller in G-J is unchanged
        return (
            self.classify_head(fused),            # [B,2]  correctness logits
            self.quality_head(fused).squeeze(1),  # [B]    quality (raw)
            self.ex_head(global_rep),             # [B,N]  exercise logits
        )

    def predict_exercise(self, kps, adj):
        self.eval()
        with torch.no_grad():
            feat_map   = self.stgcn(kps, adj)
            global_rep = self.itrans(feat_map)
            idx = int(self.ex_head(global_rep).argmax(1).item())
        return EXERCISE_LIST[idx], idx

# ─────────────────────────────────────────────────────────────────
# G. PYTORCH DATASET + DATALOADER HELPERS
# ─────────────────────────────────────────────────────────────────

class RehabDataset(Dataset):
    def __init__(self, samples, augment=False):
        self.samples = samples
        self.augment = augment

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s   = self.samples[idx]
        kps = safe_np(s['keypoints'], clip=5.0)
        if self.augment:
            kps = self._aug(kps)
        return {
            'keypoints':    torch.from_numpy(safe_np(kps, clip=5.0)).permute(2, 0, 1).float(),
            'adj':          torch.from_numpy(ADJ).float(),
            'label':        torch.tensor(s['label'],        dtype=torch.long),
            'quality':      torch.tensor(s['quality'],      dtype=torch.float32),
            'scalars':      torch.from_numpy(safe_np(s['scalars'], clip=10.0)).float(),
            'exercise_idx': torch.tensor(s['exercise_idx'], dtype=torch.long),
            'dataset_id':   torch.tensor(
                DATASET_SOURCE_MAP.get(s.get('source', 'uiprmd'), 0),
                dtype=torch.long),
            'exercise':     s['exercise'],
            'subject':      s['subject'],
        }

    def _aug(self, kps):
        kps = safe_np(kps, clip=5.0)
        if np.random.rand() < 0.5:  # temporal jitter ±20%
            n   = max(int(TARGET_LEN * np.random.uniform(0.8, 1.2)), MIN_FRAMES)
            tmp = np.zeros((n, N_JOINTS, 3), dtype=np.float32)
            for j in range(N_JOINTS):
                for c in range(3):
                    tmp[:, j, c] = _resample(kps[:, j, c], n)
            kps = np.zeros((TARGET_LEN, N_JOINTS, 3), dtype=np.float32)
            for j in range(N_JOINTS):
                for c in range(3):
                    kps[:, j, c] = _resample(tmp[:, j, c], TARGET_LEN)
        if np.random.rand() < 0.5:  # mirror flip L↔R
            kps[:, [0, 1, 2, 3, 4, 5], :] = kps[:, [3, 4, 5, 0, 1, 2], :]
            kps[:, :, 0] *= -1.0
        if np.random.rand() < 0.5:  # Gaussian noise
            kps += np.random.randn(*kps.shape).astype(np.float32) * 0.02
        if np.random.rand() < 0.3:  # small rotation in XZ plane
            th  = np.radians(np.random.uniform(-10, 10))
            x, z = kps[:, :, 0].copy(), kps[:, :, 2].copy()
            kps[:, :, 0] = np.cos(th) * x - np.sin(th) * z
            kps[:, :, 2] = np.sin(th) * x + np.cos(th) * z
        return safe_np(kps, clip=5.0)


def collate(batch):
    return {
        k: torch.stack([b[k] for b in batch])
           if isinstance(batch[0][k], torch.Tensor)
           else [b[k] for b in batch]
        for k in batch[0]
    }

def build_loaders(train_samples, test_samples, batch_size=32):
    """Build train and validation DataLoaders with exercise-balanced sampling."""
    ex_counts = Counter(s['exercise'] for s in train_samples)
    total     = len(train_samples)
    weights   = [total / ex_counts[s['exercise']] for s in train_samples]
    sampler   = WeightedRandomSampler(weights, len(train_samples), replacement=True)
    train_loader = DataLoader(
        RehabDataset(train_samples, augment=True),
        batch_size=batch_size,
        sampler=sampler,
        collate_fn=collate,
        num_workers=2,
    )
    val_loader = DataLoader(
        RehabDataset(test_samples, augment=False),
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate,
        num_workers=2,
    )
    return train_loader, val_loader


# ─────────────────────────────────────────────────────────────────
# H. TRAINING
# ─────────────────────────────────────────────────────────────────

def compute_class_weights(train_samples, device):
    """Inverse-frequency weights for cross-entropy loss."""
    n0  = sum(s['label'] == 0 for s in train_samples)
    n1  = sum(s['label'] == 1 for s in train_samples)
    tot = n0 + n1
    return torch.tensor(
        [tot / (2 * n0 + EPS), tot / (2 * n1 + EPS)],
        dtype=torch.float32, device=device,
    )

def _train_epoch(model, loader, opt, device, cw, epoch=0):
    model.train()
    A = torch.from_numpy(ADJ).to(device)
    total, used, skipped = 0.0, 0, 0

    for b in loader:
        kps    = b['keypoints'].to(device)
        lbl    = b['label'].to(device)
        qual   = b['quality'].to(device)
        sc     = b['scalars'].to(device)
        ex_idx = b['exercise_idx'].to(device)

        good = (torch.isfinite(kps).flatten(1).all(1) &
                torch.isfinite(sc).all(1) &
                torch.isfinite(qual))
        if not good.all():
            kps, lbl, qual, sc, ex_idx = (t[good] for t in
                                           (kps, lbl, qual, sc, ex_idx))
        if kps.shape[0] == 0:
            skipped += 1
            continue

        mode  = torch.zeros(kps.shape[0], dtype=torch.long, device=device)
        ds_id = b.get('dataset_id',
                      torch.zeros(kps.shape[0], dtype=torch.long)).to(device)
        if not good.all():
            ds_id = ds_id[good]
        lg, qp, ex_lg = model(kps, A, sc, mode, ex_idx, ds_id)

        if not (torch.isfinite(lg).all() and torch.isfinite(qp).all()):
            skipped += 1
            continue

        loss = (F.cross_entropy(lg, lbl, weight=cw) +
                0.3 * F.mse_loss(torch.sigmoid(qp),
                                  torch.clamp(qual, 0, 1)) +
                0.5 * F.cross_entropy(ex_lg, ex_idx))

        if not torch.isfinite(loss):
            skipped += 1
            continue

        opt.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 0.5)
        opt.step()
        total += loss.item()
        used  += 1

    if skipped:
        print(f'  ep{epoch}: skipped {skipped} batches')
    return total / max(used, 1)

@torch.no_grad()
def _evaluate(model, loader, device):
    model.eval()
    A = torch.from_numpy(ADJ).to(device)
    preds, labels, quals, qpreds = [], [], [], []

    for b in loader:
        kps    = b['keypoints'].to(device)
        sc     = b['scalars'].to(device)
        ex_idx = b['exercise_idx'].to(device)
        qual   = b['quality'].to(device)
        lbl    = b['label'].to(device)

        good = (torch.isfinite(kps).flatten(1).all(1) &
                torch.isfinite(sc).all(1))
        if good.sum() == 0:
            continue
        kps, sc, ex_idx, qual, lbl = (t[good] for t in
                                       (kps, sc, ex_idx, qual, lbl))
        mode  = torch.zeros(kps.shape[0], dtype=torch.long, device=device)
        ds_id = b.get('dataset_id',
                      torch.zeros(kps.shape[0], dtype=torch.long)).to(device)
        if good.sum() > 0 and good.sum() < ds_id.shape[0]:
            ds_id = ds_id[good.cpu()]
        lg, qp, _ = model(kps, A, sc, mode, ex_idx, ds_id)  # unpack all 3

        preds.extend(lg.argmax(1).cpu().numpy())
        labels.extend(lbl.cpu().numpy())
        quals.extend(qual.cpu().numpy())
        qpreds.extend(torch.sigmoid(qp).cpu().numpy())

    if not labels:
        return {'acc': 0.0, 'f1': 0.0, 'quality_r': 0.0}

    acc = accuracy_score(labels, preds)
    f1  = f1_score(labels, preds, average='macro', zero_division=0)
    r   = pearsonr(quals, qpreds)[0] if len(set(quals)) > 1 else 0.0
    print(classification_report(labels, preds,
          target_names=['incorrect', 'correct'], zero_division=0))
    print(f'  quality Pearson r={r:.3f}')
    return {'acc': acc, 'f1': f1, 'quality_r': r}

def run_loso(samples=None, n_epochs=N_EPOCHS,
             batch_size=32, lr=3e-4, patience=12):
    """
    Leave-One-Subject-Out cross-validation.

    Usage:
        # Option A — loads datasets internally
        results, model = run_loso()

        # Option B — pass your own sample list
        results, model = run_loso(samples=my_samples)
    """
    if samples is None:
        samples = build_master_dataset()

    device   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Device: {device}')

    subjects = get_uiprmd_subjects(samples)
    if not subjects:
        subjects = sorted({s['subject'] for s in samples})

    results = []
    best_overall_f1    = 0.0
    best_overall_model = None

    for subj in subjects:
        print(f"\n{'='*55}\nLOSO held-out: {subj}\n{'='*55}")
        tr, te = loso_split(samples, subj)
        tl, vl = build_loaders(tr, te, batch_size)
        cw     = compute_class_weights(tr, device)
        print(f'  class weights: incorrect={cw[0]:.3f}  correct={cw[1]:.3f}')

        model = RehabNet().to(device)
        opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

        def lr_fn(ep):
            w = 5
            if ep < w:
                return (ep + 1) / w
            return 0.5 * (1 + np.cos(np.pi * (ep - w) / max(n_epochs - w, 1)))

        sch = torch.optim.lr_scheduler.LambdaLR(opt, lr_fn)
        best_f1, best_state, no_imp = 0.0, None, 0

        for ep in range(n_epochs):
            loss = _train_epoch(model, tl, opt, device, cw, epoch=ep)
            sch.step()

            if (ep + 1) % 5 == 0:
                model.eval()
                pl, ll = [], []
                with torch.no_grad():
                    A2 = torch.from_numpy(ADJ).to(device)
                    for b in vl:
                        kps    = b['keypoints'].to(device)
                        sc     = b['scalars'].to(device)
                        ex_idx = b['exercise_idx'].to(device)
                        lbl    = b['label']
                        good   = (torch.isfinite(kps).flatten(1).all(1) &
                                  torch.isfinite(sc).all(1))
                        if good.sum() == 0:
                            continue
                        kps, sc, ex_idx = kps[good], sc[good], ex_idx[good]
                        lbl  = lbl[good.cpu()]
                        mode  = torch.zeros(kps.shape[0], dtype=torch.long,
                                            device=device)
                        ds_id = b.get('dataset_id',
                                      torch.zeros(kps.shape[0], dtype=torch.long)).to(device)
                        ds_id = ds_id[good.cpu()]
                        lg, _, _ = model(kps, A2, sc, mode, ex_idx, ds_id)
                        pl.extend(lg.argmax(1).cpu().numpy())
                        ll.extend(lbl.numpy())
                model.train()

                vf1 = f1_score(ll, pl, average='macro', zero_division=0)
                print(f'  ep{ep+1:3d} loss={loss:.4f}  '
                      f'lr={sch.get_last_lr()[0]:.2e}  val_f1={vf1:.3f}')

                if vf1 > best_f1:
                    best_f1    = vf1
                    best_state = copy.deepcopy(model.state_dict())
                    no_imp     = 0
                else:
                    no_imp += 1
                    if no_imp >= patience:
                        print(f'  Early stop ep{ep+1}')
                        break

        if best_state:
            model.load_state_dict(best_state)

        if best_f1 > best_overall_f1:
            best_overall_f1    = best_f1
            best_overall_model = copy.deepcopy(model.state_dict())

        print(f'\n── Eval: {subj} (best val_f1={best_f1:.3f}) ──')
        results.append(_evaluate(model, vl, device))

    # Save best checkpoint
    if best_overall_model:
        os.makedirs(os.path.dirname(MODEL_SAVE), exist_ok=True)
        torch.save(best_overall_model, MODEL_SAVE)
        print(f'\nBest model saved → {MODEL_SAVE}')

    accs = [r['acc'] for r in results]
    f1s  = [r['f1']  for r in results]
    print(f"\n{'='*55}")
    print(f'LOSO RESULTS')
    print(f'  Acc = {np.mean(accs):.3f} ± {np.std(accs):.3f}')
    print(f'  F1  = {np.mean(f1s):.3f} ± {np.std(f1s):.3f}')
    print(f"{'='*55}")

    final_model = RehabNet().to(device)
    if best_overall_model:
        final_model.load_state_dict(best_overall_model)
    return results, final_model


# ─────────────────────────────────────────────────────────────────
# I. INFERENCE + FEEDBACK
# ─────────────────────────────────────────────────────────────────

def compute_torque(quality, mode):
    """Single definition — FIX: was defined twice with different logic."""
    m = HARDWARE_MODES.get(mode, HARDWARE_MODES[0])
    return float(np.clip(m['scale'] * quality + m['bias'], m['min'], m['max']))

def generate_feedback(label, quality, scalars):
    """Priority-ordered rule engine → feedback dict."""
    s = np.asarray(scalars)
    triggered = sorted(
        [r for r in FEEDBACK_RULES if r['check'](s)],
        key=lambda r: r['priority'], reverse=True,
    )
    flags = [r['name'] for r in triggered]

    if label == 1 and quality >= 0.75:
        return {'message': 'Great rep — keep it up!',
                'short': 'Great rep', 'flags': [], 'severity': 'good'}
    if label == 1 and quality >= 0.5:
        return {'message': 'Good rep — maintain that form.',
                'short': 'Good rep', 'flags': flags, 'severity': 'good'}
    if triggered:
        top = triggered[0]
        return {'message':  top['message'],
                'short':    top['short'],
                'flags':    flags,
                'severity': 'error' if top['priority'] >= 8 else 'warning'}
    return {'message': 'Focus on smooth, controlled movement.',
            'short': 'Keep it smooth', 'flags': [], 'severity': 'warning'}

def build_hardware_payload(sample, quality, scalars, torque, mode):
    kps = sample['keypoints']

    def _ang(a, b, c):
        ba, bc = a - b, c - b
        d = np.linalg.norm(ba, axis=1) * np.linalg.norm(bc, axis=1) + EPS
        return np.degrees(np.arccos(np.clip(np.sum(ba * bc, axis=1) / d, -1, 1)))

    lk     = _ang(kps[:, 0, :], kps[:, 1, :], kps[:, 2, :])
    rk     = _ang(kps[:, 3, :], kps[:, 4, :], kps[:, 5, :])
    l_peak = float(np.min(lk));  r_peak = float(np.min(rk))
    l_rom  = float(np.max(lk) - l_peak)
    r_rom  = float(np.max(rk) - r_peak)
    ex     = sample.get('exercise', 'unknown')
    target = TARGET_ANGLES.get(ex, TARGET_ANGLES['default'])
    l_def  = max(0., target - l_peak)
    r_def  = max(0., target - r_peak)

    return {
        'torque':   round(torque, 4),
        'mode':     HARDWARE_MODES[mode]['name'],
        'mode_int': mode,
        'joint': {
            'left_knee_peak_deg':  round(l_peak, 2),
            'right_knee_peak_deg': round(r_peak, 2),
            'left_knee_rom_deg':   round(l_rom, 2),
            'right_knee_rom_deg':  round(r_rom, 2),
        },
        'target': {
            'knee_target_deg':   target,
            'left_deficit_deg':  round(l_def, 2),
            'right_deficit_deg': round(r_def, 2),
        },
        'torque_per_degree': round(torque / max(l_def, 1.0), 5),
    }

@torch.no_grad()
def infer_sample(model, sample, device=DEVICE, hardware_mode=0):
    """
    Run RehabNet on one rep.  Returns full result dict.
    FIX: model returns 3 values; always unpack with *_ or explicitly.
    """
    model.eval()
    kps_t  = (torch.from_numpy(sample['keypoints'])
              .permute(2, 0, 1).unsqueeze(0).to(device))
    adj_t  = torch.from_numpy(ADJ).to(device)
    sc_t   = torch.from_numpy(sample['scalars']).unsqueeze(0).to(device)
    mode_t = torch.tensor([hardware_mode], dtype=torch.long, device=device)
    ex_t   = torch.tensor([sample['exercise_idx']], dtype=torch.long,
                           device=device)

    ds_id = DATASET_SOURCE_MAP.get(sample.get('source', 'patient_video'), 3)
    ds_t  = torch.tensor([ds_id], dtype=torch.long, device=device)
    logits, quality, *_ = model(kps_t, adj_t, sc_t, mode_t, ex_t, ds_t)  # 3 outputs
    label   = int(logits.argmax(1).item())
    quality = float(torch.sigmoid(quality).item())
    torque  = compute_torque(quality, hardware_mode)
    fb      = generate_feedback(label, quality, sample['scalars'])
    hw      = build_hardware_payload(sample, quality, sample['scalars'],
                                     torque, hardware_mode)

    return {
        'rep_id':         sample.get('rep_id', '?'),
        'exercise':       sample.get('exercise', 'unknown'),
        'label':          label,
        'correct':        label == 1,
        'quality_score':  round(quality, 4),
        'feedback':       fb['message'],
        'feedback_short': fb['short'],
        'severity':       fb['severity'],
        'flags':          fb['flags'],
        'hardware':       hw,
        'torque_signal':  torque,
        'mode_name':      HARDWARE_MODES[hardware_mode]['name'],
        'scalars':        sample['scalars'].tolist(),
    }


# ─────────────────────────────────────────────────────────────────
# J. PATIENT SESSION — VIDEO → PER-REP FEEDBACK
# ─────────────────────────────────────────────────────────────────

def run_patient_video(video_path, patient_id,
                      model=None, exercise=None,
                      hardware_mode=0, device=DEVICE):
    """
    Full pipeline: video file → per-rep feedback dict + session summary.

    Args:
        video_path:    path to .mp4 / .webm / .avi file
        patient_id:    string identifier
        model:         trained RehabNet (if None only PS1 runs)
        exercise:      exercise name string, or None to auto-classify
        hardware_mode: 0=assist, 1=resist
        device:        torch device

    Returns: session summary dict
    """
    print(f"\n{'='*50}\nPatient: {patient_id}\n{'='*50}")

    print('\n[PS1] Processing video...')
    df = ps1_process_video(video_path, patient_id, exercise or 'unknown')

    print('\n[Bridge] Segmenting reps...')
    samples = ps1_df_to_samples(df, patient_id, exercise or 'unknown')

    if not samples:
        print('No valid reps found — check video quality')
        return {}

    if model is None:
        print('\n[PS2] No model provided — returning PS1 features only')
        return {'patient': patient_id, 'n_reps': len(samples),
                'ps1_only': True, 'samples': samples}

    # Auto-classify exercise from neural head if not given
    if exercise is None:
        kps_t = (torch.from_numpy(samples[0]['keypoints'])
                 .permute(2, 0, 1).unsqueeze(0).to(device))
        adj_t = torch.from_numpy(ADJ).to(device)
        pred_ex, pred_idx = model.predict_exercise(kps_t, adj_t)
        exercise = pred_ex
        print(f'\n[Exercise Classification] Predicted: {exercise}')
        for s in samples:
            s['exercise']     = exercise
            s['exercise_idx'] = pred_idx
    else:
        print(f'\n[Exercise] Using: {exercise}')

    print('\n[PS2] Running RehabNet inference...')
    results = []
    for s in samples:
        out    = infer_sample(model, s, device, hardware_mode)
        status = 'CORRECT ✓' if out['label'] == 1 else 'INCORRECT ✗'
        print(f"  {out['rep_id']:20s} | {status} "
              f"| quality={out['quality_score']:.2f} "
              f"| torque={out['torque_signal']:.2f} "
              f"| {out['feedback_short']}")
        results.append(out)

    n_correct = sum(r['label'] == 1 for r in results)
    avg_q     = float(np.mean([r['quality_score'] for r in results]))
    avg_t     = float(np.mean([r['torque_signal'] for r in results]))

    summary = {
        'patient':     patient_id,
        'exercise':    exercise,
        'n_reps':      len(results),
        'n_correct':   n_correct,
        'pct_correct': round(n_correct / max(len(results), 1) * 100, 1),
        'avg_quality': round(avg_q, 3),
        'avg_torque':  round(avg_t, 3),
        'hw_mode':     HARDWARE_MODES[hardware_mode]['name'],
        'reps':        results,
    }

    print(f'\nSession complete: {len(results)} reps  '
          f'correct={n_correct}/{len(results)}  '
          f'avg_quality={avg_q:.3f}')
    return summary


print('All functions loaded. ✓')
print(f'RehabNet._N_EXERCISES={RehabNet._N_EXERCISES}  '
      f'N_EXERCISES={N_EXERCISES}  '
      f'{"✓ MATCH" if RehabNet._N_EXERCISES == N_EXERCISES else "✗ MISMATCH — fix this!"}')
# -- iTransformer transplant active --
print(f'iTransformer: D_MODEL={D_MODEL}  N_HEADS={N_HEADS}  N_LAYERS={N_LAYERS}  N_DATASETS={N_DATASETS}')
_tmp = RehabNet()
_np  = sum(p.numel() for p in _tmp.parameters() if p.requires_grad)
print(f'RehabNet total trainable params: {_np:,}')
del _tmp


All functions loaded. ✓
RehabNet._N_EXERCISES=10  N_EXERCISES=10  ✓ MATCH
iTransformer: D_MODEL=128  N_HEADS=4  N_LAYERS=2  N_DATASETS=4
RehabNet total trainable params: 563,713


In [ ]:
# Check which datasets are available before starting
for name, path in [('UI-PRMD', UIPRMD_CSV),
                   ('KIMORE',  KIMORE_CSV),
                   ('Keraal',  KERAAL_CSV)]:
    status = '✓ found' if os.path.exists(path) else '✗ missing'
    print(f'{name:10s} {status}  ({path})')

# Build dataset and run LOSO training
results, trained_model = run_loso(n_epochs=60, batch_size=32, lr=3e-4)

print(f'\nBest F1: {round(max(r["f1"] for r in results), 3)}')
print(f'Model saved to: {MODEL_SAVE}')

UI-PRMD    ✓ found  (/content/drive/MyDrive/datasets/uiprmd.csv)
KIMORE     ✓ found  (/content/drive/MyDrive/datasets/kimore.csv)
Keraal     ✓ found  (/content/drive/MyDrive/datasets/keraal.csv)
UI-PRMD: 1180 samples
KIMORE: 77 samples
Keraal: 301 samples

Master dataset: 1558 samples
  correct=722  incorrect=836
  sources:   {'uiprmd', 'kimore', 'keraal'}
  exercises: {'inline_lunge', 'sit_to_stand', 'ctk_squat', 'hurdle_step', 'side_lunge', 'squat', 'deep_squat', 'straight_leg_raise'}
Device: cuda

LOSO held-out: uiprmd_s01
  class weights: incorrect=0.927  correct=1.086
  ep  5 loss=0.8291  lr=3.00e-04  val_f1=0.598
  ep 10 loss=0.7300  lr=2.94e-04  val_f1=0.679
  ep 15 loss=0.6339  lr=2.76e-04  val_f1=0.683
  ep 20 loss=0.6066  lr=2.48e-04  val_f1=0.740
  ep 25 loss=0.6215  lr=2.12e-04  val_f1=0.774
  ep 30 loss=0.5256  lr=1.71e-04  val_f1=0.758
  ep 35 loss=0.4908  lr=1.29e-04  val_f1=0.767
  ep 40 loss=0.4756  lr=8.77e-05  val_f1=0.716
  ep 45 loss=0.4317  lr=5.18e-05  val_f1=0.7

In [ ]:
# ── Load trained model ────────────────────────────────────────────
model = RehabNet().to(DEVICE)
model.load_state_dict(torch.load(MODEL_SAVE, map_location=DEVICE))
model.eval()
print(f'Model loaded from {MODEL_SAVE}')

# ── Point to your video ───────────────────────────────────────────
VIDEO_PATH = '/content/drive/MyDrive/datasets/8837221-uhd_2160_4096_25fps.mp4'

if not os.path.exists(VIDEO_PATH):
    print(f'Video not found: {VIDEO_PATH}')
    print('Upload a video and update VIDEO_PATH above')
else:
    summary = run_patient_video(
        video_path    = VIDEO_PATH,
        patient_id    = 'P001',
        model         = model,
        exercise      = None,      # None = auto-classify from neural head
        hardware_mode = 0,         # 0=assist (post-surgery), 1=resist (healthy)
        device        = DEVICE,
    )

    # Print per-rep feedback
    print('\nPer-rep breakdown:')
    for rep in summary.get('reps', []):
        deficit_l = rep['hardware']['target']['left_deficit_deg']
        deficit_r = rep['hardware']['target']['right_deficit_deg']
        print(f"  {rep['rep_id']:20s} "
              f"{'✓' if rep['correct'] else '✗'}  "
              f"q={rep['quality_score']:.2f}  "
              f"torque={rep['torque_signal']:.3f}  "
              f"deficit L={deficit_l}° R={deficit_r}°  "
              f"\"{rep['feedback_short']}\"")

Model loaded from /content/drive/MyDrive/models/rehabnet_best.pth

Patient: P001

[PS1] Processing video...
  Downloaded.
  PS1: 639 frames  detection=100%  62.2s

[Bridge] Segmenting reps...
  Segmenter: n_frames=639  min_dist=50  peaks=7
  Bridge: 7 rep segments → 7 valid

[Exercise Classification] Predicted: straight_leg_raise

[PS2] Running RehabNet inference...
  P001_r1              | CORRECT ✓ | quality=0.67 | torque=0.30 | Good rep
  P001_r2              | CORRECT ✓ | quality=0.54 | torque=0.34 | Good rep
  P001_r3              | INCORRECT ✗ | quality=0.47 | torque=0.36 | Bend knee further
  P001_r4              | INCORRECT ✗ | quality=0.23 | torque=0.43 | Bend knee further
  P001_r5              | CORRECT ✓ | quality=0.91 | torque=0.23 | Great rep
  P001_r6              | INCORRECT ✗ | quality=0.22 | torque=0.43 | Bend knee further
  P001_r7              | CORRECT ✓ | quality=0.67 | torque=0.30 | Good rep

Session complete: 7 reps  correct=4/7  avg_quality=0.529

Per-rep break